# Importing Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path

In [1]:
Path = ("/Users/mehreenwerth/Desktop/citibike-weather-2022")

# Sample from Raw Files

In [3]:
np.random.seed(32)

RAW_DIR = Path("data_raw/csv")
files = sorted(list(RAW_DIR.glob("*.csv"))) + sorted(list(RAW_DIR.glob("*.csv.gz")))

keep_cols = [
    "rideable_type","started_at","ended_at",
    "start_station_name","end_station_name",
    "start_lat","start_lng","end_lat","end_lng",
    "member_casual"
]

PER_FILE_N = 5000

samples = []
for f in files:
    df = pd.read_csv(f, usecols=keep_cols, low_memory=False)
    n = min(PER_FILE_N, len(df))
    samples.append(df.sample(n=n, random_state=32))

df_sample = pd.concat(samples, ignore_index=True)
df_sample.shape

(180000, 10)

# Adding Weather

In [4]:
weather = pd.read_csv("data_processed/weather_lga_2022.csv")
weather["date"] = pd.to_datetime(weather["date"])
weather["TAVG_C"] = weather["TAVG"] / 10

df_sample["started_at"] = pd.to_datetime(df_sample["started_at"], errors="coerce")
df_sample["date"] = df_sample["started_at"].dt.floor("D")

df_sample = df_sample.merge(weather[["date","TAVG_C","PRCP","AWND"]], on="date", how="left")

# Saving Sample

In [5]:
from pathlib import Path
out_path = Path("df_sample_seed32.csv")
df_sample.to_csv(out_path, index=False)

out_path, out_path.stat().st_size / (1024*1024)

(PosixPath('df_sample_seed32.csv'), 32.76012706756592)